In [22]:
import torch
import torch.nn as nn
import math
import pandas as pd
import numpy as np
from models import LanguageModel

okay
okay
okay
okay


In [47]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='mps')

In [2]:
with open('../data/input.txt', 'r') as f:
    text = f.read()

In [3]:
text[:500]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor"

In [4]:
len(set(text))

65

In [5]:
stoi = {}
itos = {}

In [6]:
i = 0
for char in text:
    if char not in stoi:
        stoi[char] = i
        itos[i] = char
        i += 1

In [25]:
vocab_size=len(stoi)
vocab_size

65

In [8]:
stoi['n']

9

In [9]:
itos[9]

'n'

In [10]:
stoi['\n']

11

In [11]:
itos[11]

'\n'

In [12]:
encoded = torch.tensor([stoi[char] for char in text])
decoded = ''.join(itos[i.item()] for i in encoded[:100])

In [13]:
decoded

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [14]:
text[0:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [15]:
encoded[:100]

tensor([ 0,  1,  2,  3,  4,  5,  6,  1,  4,  1,  7,  8,  9, 10, 11, 12,  8, 13,
        14,  2,  8,  5, 15,  8,  5, 16,  2, 14, 17,  8,  8, 18,  5, 19,  9, 20,
         5, 13, 21,  2,  4, 22,  8,  2, 23,  5, 22,  8, 19,  2,  5, 24,  8,  5,
         3, 16,  8, 19, 25, 26, 11, 11, 27, 28, 28, 10, 11, 29, 16,  8, 19, 25,
        23,  5,  3, 16,  8, 19, 25, 26, 11, 11,  0,  1,  2,  3,  4,  5,  6,  1,
         4,  1,  7,  8,  9, 10, 11, 30, 14, 21])

In [16]:
train = encoded[:int((0.9) * len(text))]
test = encoded[int((0.9) * len(text)):]

In [17]:
len(train)

1003854

In [18]:
len(test)

111540

In [30]:
batchsize = 4
blocksize = 8
torch.manual_seed(1337)


def make_batch(blocksize=blocksize,batchsize=batchsize,_train=True):
    if _train:
        data=train
    else:
        data=test
    indexx = torch.randint(len(data) - blocksize, (batchsize,))
    x = torch.stack([data[i:i + blocksize] for i in indexx])
    y = torch.stack([data[i + 1:i + blocksize + 1] for i in indexx])
    return x, y


In [31]:
xb, yb = make_batch()
xb.shape
yb.shape

torch.Size([4, 8])

In [32]:
def count_non_embedding_params(model):
    return sum(p.numel() for name, p in model.named_parameters()
               if 'embed' not in name and 'lm_head' not in name)

In [23]:
#Parameter Scaling Sweep

In [26]:
model1 = LanguageModel(d_model=64,num_layers=2,num_heads=2,blocksize=128,vocab_size=vocab_size)
model2 = LanguageModel(d_model=128,num_layers=4,num_heads=4,blocksize=256,vocab_size=vocab_size)
model3 = LanguageModel(d_model=256,num_layers=6,num_heads=8,blocksize=256,vocab_size=vocab_size)

In [27]:
count_non_embedding_params(model1)

112448

In [28]:
count_non_embedding_params(model2)

834432

In [29]:
count_non_embedding_params(model3)

4821248

In [48]:
@torch.no_grad()
def get_val_los(model,blocksize,batchsize, iter):
    model.eval()
    losses=[]
    for i in range(iter):
        x,y = make_batch(blocksize=blocksize,batchsize=batchsize,_train=False)
        x=x.to(device)
        y=y.to(device)
        logits = model(x)
        B, T, C = logits.shape
        loss = torch.nn.functional.cross_entropy(logits.view(B*T,C), y.view(B*T))
        losses.append(loss.item())
    return sum(losses)/len(losses)


In [55]:
def train_sweep(model, blocksize, batchsize=32, steps=3000):
    model=model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3)
    best_los_val = float('inf')

    for step in range(steps):
        x,y = make_batch(blocksize=blocksize,batchsize=batchsize)
        x=x.to(device)
        y=y.to(device)
        logits = model(x)
        B, T, C = logits.shape
        loss = torch.nn.functional.cross_entropy(logits.view(B*T,C), y.view(B*T))
        optim.zero_grad(set_to_none=True)
        loss.backward()
        optim.step()
        if step%200==0 or step==steps-1:
            val_loss=get_val_los(model,blocksize,batchsize,50)
            model.train()
            best_los_val = min(best_los_val, val_loss)
            print(f'Step : {step+1}, Train loss: {loss.item():.4f}, best val loss: {best_los_val:.4f}')
    return best_los_val

In [44]:
best_val_loss_1 = train_sweep(model1,blocksize=128)

Step : 1, Train loss: 41.9660, best val loss: 39.9687
Step : 201, Train loss: 2.9741, best val loss: 2.9886
Step : 401, Train loss: 2.7868, best val loss: 2.7601
Step : 601, Train loss: 2.6158, best val loss: 2.6413
Step : 801, Train loss: 2.5453, best val loss: 2.5771
Step : 1001, Train loss: 2.5135, best val loss: 2.5488
Step : 1201, Train loss: 2.4829, best val loss: 2.5193
Step : 1401, Train loss: 2.4296, best val loss: 2.4851
Step : 1601, Train loss: 2.3854, best val loss: 2.4134
Step : 1801, Train loss: 2.3417, best val loss: 2.3496
Step : 2001, Train loss: 2.2702, best val loss: 2.2683
Step : 2201, Train loss: 2.1457, best val loss: 2.2199
Step : 2401, Train loss: 2.1189, best val loss: 2.1723
Step : 2601, Train loss: 2.1418, best val loss: 2.1382
Step : 2801, Train loss: 2.0442, best val loss: 2.1240
Step : 3000, Train loss: 2.0001, best val loss: 2.1035


In [45]:
best_val_loss_2 = train_sweep(model2,blocksize=256)

Step : 1, Train loss: 82.9142, best val loss: 70.9591
Step : 201, Train loss: 2.7399, best val loss: 2.7105
Step : 401, Train loss: 2.5661, best val loss: 2.5736
Step : 601, Train loss: 2.5213, best val loss: 2.5362
Step : 801, Train loss: 2.4791, best val loss: 2.5154
Step : 1001, Train loss: 2.4315, best val loss: 2.4578
Step : 1201, Train loss: 2.3689, best val loss: 2.3836
Step : 1401, Train loss: 2.2073, best val loss: 2.2731
Step : 1601, Train loss: 2.1474, best val loss: 2.1958
Step : 1801, Train loss: 2.0532, best val loss: 2.1375
Step : 2001, Train loss: 2.0015, best val loss: 2.0798
Step : 2201, Train loss: 1.9287, best val loss: 2.0436
Step : 2401, Train loss: 1.8655, best val loss: 1.9869
Step : 2601, Train loss: 1.7934, best val loss: 1.9450
Step : 2801, Train loss: 1.7044, best val loss: 1.9184
Step : 3000, Train loss: 1.7190, best val loss: 1.8751


In [56]:
best_val_loss_3 = train_sweep(model3,blocksize=256)

Step : 1, Train loss: 1.9313, best val loss: 6.7833
Step : 201, Train loss: 1.9211, best val loss: 2.0430
Step : 401, Train loss: 1.8008, best val loss: 1.9726
Step : 601, Train loss: 1.7093, best val loss: 1.8922
Step : 801, Train loss: 1.6943, best val loss: 1.8404
Step : 1001, Train loss: 1.6107, best val loss: 1.8166
Step : 1201, Train loss: 1.5287, best val loss: 1.7880
Step : 1401, Train loss: 1.5376, best val loss: 1.7701
Step : 1601, Train loss: 1.4479, best val loss: 1.7317
Step : 1801, Train loss: 1.4280, best val loss: 1.6995
Step : 2001, Train loss: 1.4208, best val loss: 1.6995
Step : 2201, Train loss: 1.3561, best val loss: 1.6728
Step : 2401, Train loss: 1.4078, best val loss: 1.6547
Step : 2601, Train loss: 1.3359, best val loss: 1.6547
Step : 2801, Train loss: 1.3481, best val loss: 1.6547
Step : 3000, Train loss: 1.3110, best val loss: 1.6547


In [58]:
#Data Scaling Sweep
_model2 = LanguageModel(d_model=128,num_layers=4,num_heads=4,blocksize=256,vocab_size=vocab_size)

In [63]:
def make_batch_for_scaling(blocksize,batchsize,split=1,is_train=True):
    if is_train:
        data=train[:int(split*len(train))]
    else:
        data=test
    indexx = torch.randint(len(data) - blocksize, (batchsize,))
    x = torch.stack([data[i:i + blocksize] for i in indexx])
    y = torch.stack([data[i + 1:i + blocksize + 1] for i in indexx])
    return x, y

In [64]:
def train_data_scaling_sweep(model, blocksize, batchsize=32, steps=3000, split=1):
    model=model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3)
    best_los_val = float('inf')

    for step in range(steps):
        x,y = make_batch_for_scaling(blocksize=blocksize,batchsize=batchsize, split=split)
        x=x.to(device)
        y=y.to(device)
        logits = model(x)
        B, T, C = logits.shape
        loss = torch.nn.functional.cross_entropy(logits.view(B*T,C), y.view(B*T))
        optim.zero_grad(set_to_none=True)
        loss.backward()
        optim.step()
        if step%200==0 or step==steps-1:
            val_loss=get_val_los(model,blocksize,batchsize,50)
            model.train()
            best_los_val = min(best_los_val, val_loss)
            print(f'Step : {step+1}, Train loss: {loss.item():.4f}, best val loss: {best_los_val:.4f}')
    return best_los_val

In [65]:
best_val_loss_scalling_10 = train_data_scaling_sweep(_model2,256, 32,split=0.1)

Step : 1, Train loss: 81.8781, best val loss: 70.9437
Step : 201, Train loss: 2.6365, best val loss: 2.7856
Step : 401, Train loss: 2.4608, best val loss: 2.7063
Step : 601, Train loss: 2.4238, best val loss: 2.6786
Step : 801, Train loss: 2.3946, best val loss: 2.6786
Step : 1001, Train loss: 2.3132, best val loss: 2.6534
Step : 1201, Train loss: 2.2036, best val loss: 2.6463
Step : 1401, Train loss: 2.0770, best val loss: 2.6093
Step : 1601, Train loss: 1.9356, best val loss: 2.6093
Step : 1801, Train loss: 1.8789, best val loss: 2.6093
Step : 2001, Train loss: 1.7073, best val loss: 2.6093
Step : 2201, Train loss: 1.6739, best val loss: 2.6093
Step : 2401, Train loss: 1.5498, best val loss: 2.6093
Step : 2601, Train loss: 1.4909, best val loss: 2.6093
Step : 2801, Train loss: 1.4784, best val loss: 2.6093
Step : 3000, Train loss: 1.3469, best val loss: 2.6093


In [73]:
_model2 = LanguageModel(d_model=128,num_layers=4,num_heads=4,blocksize=256,vocab_size=vocab_size)
best_val_loss_scalling_25 = train_data_scaling_sweep(_model2,256, 32,split=0.25)

Step : 1, Train loss: 82.8262, best val loss: 72.3329
Step : 201, Train loss: 2.6727, best val loss: 2.7515
Step : 401, Train loss: 2.5321, best val loss: 2.6184
Step : 601, Train loss: 2.4320, best val loss: 2.5945
Step : 801, Train loss: 2.4330, best val loss: 2.5675
Step : 1001, Train loss: 2.4407, best val loss: 2.5378
Step : 1201, Train loss: 2.2829, best val loss: 2.4233
Step : 1401, Train loss: 2.1107, best val loss: 2.3646
Step : 1601, Train loss: 1.9934, best val loss: 2.3646
Step : 1801, Train loss: 1.9596, best val loss: 2.3646
Step : 2001, Train loss: 1.8834, best val loss: 2.3646
Step : 2201, Train loss: 1.8393, best val loss: 2.3646
Step : 2401, Train loss: 1.8163, best val loss: 2.3646
Step : 2601, Train loss: 1.6600, best val loss: 2.3646
Step : 2801, Train loss: 1.6472, best val loss: 2.3646
Step : 3000, Train loss: 1.5994, best val loss: 2.3646


In [74]:
_model2 = LanguageModel(d_model=128,num_layers=4,num_heads=4,blocksize=256,vocab_size=vocab_size)
best_val_loss_scalling_50 = train_data_scaling_sweep(_model2,256, 32,split=0.5)

Step : 1, Train loss: 84.1086, best val loss: 72.3706
Step : 201, Train loss: 2.7264, best val loss: 2.7556
Step : 401, Train loss: 2.5352, best val loss: 2.5954
Step : 601, Train loss: 2.5137, best val loss: 2.5532
Step : 801, Train loss: 2.4730, best val loss: 2.5352
Step : 1001, Train loss: 2.4352, best val loss: 2.5124
Step : 1201, Train loss: 2.4304, best val loss: 2.5098
Step : 1401, Train loss: 2.4099, best val loss: 2.4759
Step : 1601, Train loss: 2.2429, best val loss: 2.3638
Step : 1801, Train loss: 2.1302, best val loss: 2.2760
Step : 2001, Train loss: 2.0289, best val loss: 2.2219
Step : 2201, Train loss: 1.9429, best val loss: 2.1889
Step : 2401, Train loss: 1.8925, best val loss: 2.1889
Step : 2601, Train loss: 1.8444, best val loss: 2.1837
Step : 2801, Train loss: 1.7290, best val loss: 2.1692
Step : 3000, Train loss: 1.7123, best val loss: 2.1380


In [75]:
_model2 = LanguageModel(d_model=128,num_layers=4,num_heads=4,blocksize=256,vocab_size=vocab_size)
best_val_loss_scalling_100 = train_data_scaling_sweep(_model2,256, 32,split=1)

Step : 1, Train loss: 83.6373, best val loss: 72.6224
Step : 201, Train loss: 2.7200, best val loss: 2.7379
Step : 401, Train loss: 2.5251, best val loss: 2.5812
Step : 601, Train loss: 2.5324, best val loss: 2.5256
Step : 801, Train loss: 2.4788, best val loss: 2.4957
Step : 1001, Train loss: 2.4464, best val loss: 2.4650
Step : 1201, Train loss: 2.3164, best val loss: 2.3628
Step : 1401, Train loss: 2.1805, best val loss: 2.2490
Step : 1601, Train loss: 2.1127, best val loss: 2.1642
Step : 1801, Train loss: 2.0232, best val loss: 2.1123
Step : 2001, Train loss: 1.9596, best val loss: 2.0757
Step : 2201, Train loss: 1.9434, best val loss: 2.0464
Step : 2401, Train loss: 1.8150, best val loss: 1.9927
Step : 2601, Train loss: 1.7909, best val loss: 1.9387
Step : 2801, Train loss: 1.7719, best val loss: 1.9224
Step : 3000, Train loss: 1.7048, best val loss: 1.8820


In [77]:
best_val_loss_1

2.103500528335571

In [78]:
best_val_loss_2

1.8751439380645751

In [79]:
best_val_loss_3

1.6547045993804932

In [80]:
print(best_val_loss_scalling_10)
print(best_val_loss_scalling_25)
print(best_val_loss_scalling_50)
print(best_val_loss_scalling_100)

2.6093492794036863
2.364648847579956
2.138003306388855
1.8820109105110168


In [81]:
for name, _ in model1.named_parameters():
    print(name)

token_emb.weight
pos_emb.emb.weight
transformers.0.ff.sub_layer.layer1.weight
transformers.0.ff.sub_layer.layer1.bias
transformers.0.ff.sub_layer.layer2.weight
transformers.0.ff.sub_layer.layer2.bias
transformers.0.ff.norm.weight
transformers.0.ff.norm.bias
transformers.0.att.sub_layer.query.weight
transformers.0.att.sub_layer.query.bias
transformers.0.att.sub_layer.key.weight
transformers.0.att.sub_layer.key.bias
transformers.0.att.sub_layer.value.weight
transformers.0.att.sub_layer.value.bias
transformers.0.att.sub_layer.output.weight
transformers.0.att.sub_layer.output.bias
transformers.0.att.norm.weight
transformers.0.att.norm.bias
transformers.1.ff.sub_layer.layer1.weight
transformers.1.ff.sub_layer.layer1.bias
transformers.1.ff.sub_layer.layer2.weight
transformers.1.ff.sub_layer.layer2.bias
transformers.1.ff.norm.weight
transformers.1.ff.norm.bias
transformers.1.att.sub_layer.query.weight
transformers.1.att.sub_layer.query.bias
transformers.1.att.sub_layer.key.weight
transformers

In [82]:
print(best_val_loss_1)
print(best_val_loss_2)
print(best_val_loss_3)

print(best_val_loss_scalling_10)
print(best_val_loss_scalling_25)
print(best_val_loss_scalling_50)
print(best_val_loss_scalling_100)

print(count_non_embedding_params(model1))
print(count_non_embedding_params(model2))
print(count_non_embedding_params(model3))

2.103500528335571
1.8751439380645751
1.6547045993804932
2.6093492794036863
2.364648847579956
2.138003306388855
1.8820109105110168
112448
834432
4821248
